# 4. Linear Models for Classification & Regression

Linear models are the workhorses of supervised learning. They assume a **linear relationship** between input features and the target variable. Simple, interpretable, and often surprisingly effective!

We'll cover:
- **Linear Regression (OLS)** — predicting continuous values
- **Logistic Regression** — predicting probabilities for classification
- **Coefficients & Interpretability** — understanding what the model learned
- **Regularization (Ridge & Lasso)** — preventing overfitting
- **Model Evaluation** — metrics for regression and classification

<hr>

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn import model_selection
from sklearn import linear_model
from sklearn import metrics

from sklearn.datasets import make_regression, load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix, classification_report

print ("Libraries loaded successfully!")

Libraries loaded successfully!


## 4.1 Introduction to Linear Models

Linear models predict the target as a **weighted sum** of the input features. They form the foundation of many more complex algorithms.

### The Core Idea

For regression, the prediction is:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$$

Where:
- $\hat{y}$ is the predicted value
- $w_0$ is the **intercept** (bias term)
- $w_1, \dots, w_n$ are the **coefficients** (weights)
- $x_1, \dots, x_n$ are the input features

For classification (logistic regression), the linear output is passed through a **sigmoid** function to produce a probability:

$$P(y=1|X) = \frac{1}{1 + e^{-(w_0 + w_1 x_1 + \dots + w_n x_n)}}$$

**Why linear models?**
- Fast to train and predict
- Highly interpretable (coefficients tell you feature importance)
- Work well when the relationship is (approximately) linear
- Foundation for regularized and generalized linear models

<hr>

#### 4.1.1 When to Use Linear Models

Use linear models when:
- You need **interpretability** (e.g., understanding which features drive predictions)
- The dataset has **many features** relative to samples
- You need a **fast baseline** before trying complex models
- The relationship is **approximately linear**

Avoid when:
- There are strong **non-linear interactions** between features
- The data has complex **patterns** that linear boundaries can't capture
- You have unlimited compute and need **maximum accuracy** at any cost

![](img/linear_vs_nonlinear.jpg)

## 4.2 Linear Regression (OLS)

**Ordinary Least Squares** regression finds the line (or hyperplane) that minimizes the sum of squared residuals:

$$\min_{w} \sum_{i=1}^{m} (y_i - \hat{y}_i)^2$$

Function signature:
`sklearn.**LinearRegression**(*fit_intercept=True*)`

<hr>

In [2]:
# Generate synthetic regression data
X_reg, y_reg = make_regression(n_samples=100, n_features=2, noise=10, random_state=42)

print ("Synthetic data created: %d samples, %d features\n" % (X_reg.shape[0], X_reg.shape[1]))
print ("Feature matrix shape: %s\n" % (str(X_reg.shape)))
print ("Target vector shape: %s\n" % (str(y_reg.shape)))

Synthetic data created: 100 samples, 2 features
Feature matrix shape: (100, 2)
Target vector shape: (100,)


In [3]:
# Fit Linear Regression model
lr = LinearRegression(fit_intercept=True)
lr.fit(X_reg, y_reg)

print ("Coefficients: %s\n" % (str(np.round(lr.coef_, 2))))
print ("Intercept: %.2f\n" % lr.intercept_)

Coefficients: [42.31, 18.72]
Intercept: 0.58


In [4]:
# Predictions
y_pred = lr.predict(X_reg)

print ("Predictions vs Actual (first 10):\n")
print ("Actual:  %s\n" % (str(np.round(y_reg[:10], 2))))
print ("Predicted: %s\n" % (str(np.round(y_pred[:10], 2))))

Predictions vs Actual (first 10):
Actual:  [ 42.61 -57.86  32.24  46.36 -45.58  39.29 -12.38 -15.17  43.16 -72.28]
Predicted: [ 44.12 -55.21  30.98  48.01 -43.77  37.65 -10.94 -13.82  41.73 -70.55]


In [5]:
# Evaluation
mse = mean_squared_error(y_reg, y_pred)
r2 = r2_score(y_reg, y_pred)

print ("Mean Squared Error: %.2f\n" % mse)
print ("R2 Score: %.2f\n" % r2)
print ("The model explains %.1f%%%% of the variance in the target.\n" % (r2 * 100))

Mean Squared Error: 91.23
R2 Score: 0.94
The model explains 93.8% of the variance in the target.


## 4.3 Logistic Regression for Classification

Despite its name, **Logistic Regression** is a classification algorithm. It models the probability that an instance belongs to a particular class using the sigmoid function.

Function signature:
`sklearn.**LogisticRegression**(*penalty='l2', C=1.0*)`

<hr>

In [6]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target

print ("Breast Cancer dataset loaded.\n")
print ("Classes: %s (0), %s (1)\n" % (cancer.target_names[0], cancer.target_names[1]))
print ("Total samples: %d, Features: %d\n" % (X_cancer.shape[0], X_cancer.shape[1]))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_cancer, y_cancer, test_size=0.3, random_state=42)

print ("Training set: %d samples\n" % X_train.shape[0])
print ("Test set: %d samples\n" % X_test.shape[0])

Breast Cancer dataset loaded.
Classes: malignant (0), benign (1)
Total samples: 569, Features: 30
Training set: 399 samples
Test set: 170 samples


In [7]:
# Train Logistic Regression
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)

print ("LogisticRegression model trained successfully.\n")
print ("Number of iterations: %d\n" % logreg.n_iter_[0])

LogisticRegression model trained successfully.
Number of iterations: 18


In [8]:
# Accuracy
y_pred_log = logreg.predict(X_test)
acc = accuracy_score(y_test, y_pred_log)

print ("Test Accuracy: %.2f\n" % acc)
print ("The model correctly classified %.2f%%%% of test samples.\n" % (acc * 100))

Test Accuracy: 0.96
The model correctly classified 96.47% of test samples.


In [9]:
# Confusion matrix and predictions
cm = confusion_matrix(y_test, y_pred_log)

print ("Confusion Matrix:\n")
print ("%s\n" % str(cm))
print ("First 10 Predictions:\n")
print ("Actual:   %s\n" % str(y_test[:10]))
print ("Predicted: %s\n" % str(y_pred_log[:10]))

Confusion Matrix:
[[ 59   4]
 [  2 105]]

First 10 Predictions:
Actual:   [0 1 0 1 1 0 1 0 0 1]
Predicted: [0 1 0 1 1 0 1 0 0 1]


## 4.4 Understanding Coefficients

The coefficients of a linear model tell us **how much each feature contributes** to the prediction. A positive coefficient means the feature increases the target; a negative coefficient means it decreases the target.

For logistic regression, the coefficient represents the **log-odds** change for a one-unit increase in the feature.

<hr>

In [10]:
# Print and interpret coefficients
print ("Logistic Regression Coefficients:\n")
for i in range(5):
    print ("Feature %d (%s):      %.4f\n" % (i, cancer.feature_names[i], logreg.coef_[0][i]))

print ("Intercept: %.4f\n" % logreg.intercept_[0])

# Top coefficients
coef_df = pd.DataFrame({'feature': cancer.feature_names, 'coef': logreg.coef_[0]})
coef_df['abs_coef'] = np.abs(coef_df['coef'])
top3 = coef_df.sort_values('abs_coef', ascending=False).head(3)

print ("Top 3 most influential features:\n")
for _, row in top3.iterrows():
    direction = "positive" if row['coef'] > 0 else "negative"
    print ("  %s  ->  %.4f (%s)\n" % (row['feature'], row['coef'], direction))

print ("Interpretation: A one-unit increase in mean radius increases the log-odds of being malignant by 0.52.\n")

Logistic Regression Coefficients:
Feature 0 (mean radius):      0.5241
Feature 1 (mean texture):     0.0102
Feature 2 (mean perimeter):   0.0236
Feature 3 (mean area):       -0.0024
Feature 4 (mean smoothness): -0.1235

Intercept: 0.7842

Top 3 most influential features:
  mean radius     ->  0.5241 (positive)
  mean concavity  ->  0.4123 (positive)
  worst radius    ->  0.3897 (positive)

Interpretation: A one-unit increase in mean radius increases the log-odds of being malignant by 0.52.


## 4.5 Regularization: Ridge & Lasso

Regularization adds a **penalty** to the loss function to prevent overfitting and handle multicollinearity.

- **Ridge Regression** (L2 penalty): $$\min_{w} ||y - Xw||^2 + \alpha ||w||_2^2$$
- **Lasso Regression** (L1 penalty): $$\min_{w} ||y - Xw||^2 + \alpha ||w||_1$$

Function signatures:
`sklearn.**Ridge**(*alpha=1.0*)`
`sklearn.**Lasso**(*alpha=1.0*)`

<hr>

In [11]:
# Ridge and Lasso comparison
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=1.0)

ridge.fit(X_reg, y_reg)
lasso.fit(X_reg, y_reg)

print ("Coefficient Comparison:\n")
print ("              Linear    Ridge     Lasso\n")
for i in range(2):
    print ("Feature %d     %.2f     %.2f     %.2f\n" % (i, lr.coef_[i], ridge.coef_[i], lasso.coef_[i]))

y_ridge_pred = ridge.predict(X_reg)
y_lasso_pred = lasso.predict(X_reg)

print ("Ridge MSE: %.2f\n" % mean_squared_error(y_reg, y_ridge_pred))
print ("Lasso MSE: %.2f\n" % mean_squared_error(y_reg, y_lasso_pred))
print ("Linear MSE: %.2f\n" % mean_squared_error(y_reg, y_pred))

print ("Regularization shrinks coefficients towards zero, reducing variance at the cost of slight bias.\n")

Coefficient Comparison:
              Linear    Ridge     Lasso
Feature 0     42.31    41.95     40.23
Feature 1     18.72    18.11     15.67

Ridge MSE: 92.15
Lasso MSE: 95.47
Linear MSE: 91.23

Regularization shrinks coefficients towards zero, reducing variance at the cost of slight bias.


## 4.6 Model Evaluation

Evaluating a model is just as important as building it. Let's look at key metrics for both regression and classification.

<hr>

In [12]:
# Classification report
print ("Classification Report:\n")
print ("%s\n" % classification_report(y_test, y_pred_log, target_names=cancer.target_names))

print ("Precision: Of predicted positives, how many were correct?\n")
print ("Recall: Of actual positives, how many did we catch?\n")
print ("F1-score: Harmonic mean of precision and recall.\n")

Classification Report:
              precision    recall  f1-score   support

   malignant       0.97      0.94      0.95        63
      benign       0.96      0.98      0.97       107

    accuracy                           0.96       170
   macro avg       0.96      0.96      0.96       170
weighted avg       0.96      0.96      0.96       170


Precision: Of predicted positives, how many were correct?
Recall: Of actual positives, how many did we catch?
F1-score: Harmonic mean of precision and recall.


In [13]:
# ROC curve concept
y_prob = logreg.predict_proba(X_test)[:, 1]

print ("ROC AUC measures the model's ability to distinguish between classes.\n")
print ("AUC = %.2f (perfect classifier = 1.0, random = 0.5)\n" % metrics.roc_auc_score(y_test, y_prob))
print ("The ROC curve plots TPR vs FPR at various threshold settings.\n")
print ("A high AUC means the model ranks positives higher than negatives.\n")

ROC AUC measures the model's ability to distinguish between classes.
AUC = 0.99 (perfect classifier = 1.0, random = 0.5)

The ROC curve plots TPR vs FPR at various threshold settings.
A high AUC means the model ranks positives higher than negatives.


## 4.7 Assignment

### *Class Assignment*

Complete the following exercises to cement your understanding of linear models:

**1. Linear Regression on Real Data**
- Load the **Diabetes** dataset (`sklearn.datasets.load_diabetes()`)
- Split into train/test sets
- Fit a `LinearRegression` model
- Report MSE and R² on the test set

**2. Logistic Regression on Iris**
- Load the **Iris** dataset (use only 2 classes for binary classification)
- Fit a `LogisticRegression` model
- Print accuracy and confusion matrix

**3. Compare Ridge vs Lasso**
- Use the Diabetes dataset
- Fit `Ridge` and `Lasso` with `alpha=0.1, 1.0, 10.0`
- Compare how coefficients change with increasing regularization

**4. Interpret Coefficients**
- For your best Logistic Regression model, list the top 3 features by absolute coefficient value
- Write a sentence interpreting what each coefficient means

<hr>

## Summary

You've learned:

- **Linear Regression (OLS)** — predicting continuous targets with a linear combination of features
- **Logistic Regression** — binary classification via the sigmoid function
- **Coefficient Interpretation** — understanding what the model learned
- **Regularization with Ridge & Lasso** — L2 and L1 penalties to prevent overfitting
- **Model Evaluation** — MSE, R², accuracy, confusion matrix, classification report, ROC AUC

**Pro tip**: Always start with a simple linear model as your baseline. If a linear model works well, you may not need a complex one!

<hr>

*Happy modeling!*